# 03 — Twitch Baseline Ranker

Train strong, interpretable baselines on the ranking tables produced by Notebook 02.

We compare:
- historical streamer popularity
- logistic regression
- histogram gradient boosting

The models output **inference scores** for each `(user, candidate streamer)` pair. Ranking is then performed by sorting those scores within each user's candidate set.

> The candidate sets in this notebook contain sampled negatives, so ranking metrics measure ranking quality on the sampled evaluation set rather than full-catalog retrieval quality.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from time import perf_counter

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib

PROJECT_DIR = Path("Distributed ML Inference & Ranking")
DATA_DIR = PROJECT_DIR / "data/processed/twitch"
ARTIFACT_DIR = PROJECT_DIR / "artifacts/twitch"
PRED_DIR = PROJECT_DIR / "data/processed/twitch/predictions"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "twitch_train_ranking.csv.gz"
TEST_PATH = DATA_DIR / "twitch_test_ranking.csv.gz"

## 1. Load train/test ranking tables

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test: ", test.shape)
print("Train positive rate:", f"{train['label'].mean():.2%}")
print("Test positive rate: ", f"{test['label'].mean():.2%}")

train.head()

## 2. Define feature columns

In [ ]:
ID_COLUMNS = ["user_id", "streamer"]
LABEL_COLUMN = "label"
FUTURE_ONLY_COLUMNS = ["future_interactions", "future_watch_minutes"]

FEATURE_COLUMNS = [
    c for c in train.columns
    if c not in ID_COLUMNS + [LABEL_COLUMN] + FUTURE_ONLY_COLUMNS
]

print(f"Number of features: {len(FEATURE_COLUMNS)}")
FEATURE_COLUMNS

## 3. Ranking metrics

In [ ]:
def ranking_metrics(frame, score_col, ks=(5, 10, 20)):
    ranked = frame.sort_values(
        ["user_id", score_col],
        ascending=[True, False],
    ).copy()

    rows = []

    for user_id, group in ranked.groupby("user_id", sort=False):
        labels = group["label"].to_numpy()
        positives = labels.sum()

        if positives == 0:
            continue

        result = {"user_id": user_id}

        positive_positions = np.flatnonzero(labels == 1)
        first_positive_rank = positive_positions[0] + 1
        result["reciprocal_rank"] = 1.0 / first_positive_rank

        for k in ks:
            top = labels[:k]
            hits = top.sum()

            result[f"recall@{k}"] = hits / positives
            result[f"hitrate@{k}"] = float(hits > 0)

            discounts = 1.0 / np.log2(np.arange(2, len(top) + 2))
            dcg = (top * discounts).sum()

            ideal_len = min(int(positives), k)
            if ideal_len > 0:
                ideal_discounts = 1.0 / np.log2(
                    np.arange(2, ideal_len + 2)
                )
                idcg = ideal_discounts.sum()
                result[f"ndcg@{k}"] = dcg / idcg
            else:
                result[f"ndcg@{k}"] = np.nan

        rows.append(result)

    per_user = pd.DataFrame(rows)

    summary = {
        "model": score_col,
        "MRR": per_user["reciprocal_rank"].mean(),
    }

    for k in ks:
        summary[f"Recall@{k}"] = per_user[f"recall@{k}"].mean()
        summary[f"HitRate@{k}"] = per_user[f"hitrate@{k}"].mean()
        summary[f"NDCG@{k}"] = per_user[f"ndcg@{k}"].mean()

    return pd.Series(summary), per_user

## 4. Popularity baseline

In [ ]:
test_predictions = test[
    ["user_id", "streamer", "label", "streamer_interactions", "seen_before"]
].copy()

# Historical popularity is already leakage-safe because it was built
# from behavior before the test prediction cutoff.
test_predictions["score_popularity"] = test["streamer_interactions"].fillna(0)

pop_summary, _ = ranking_metrics(
    test_predictions,
    "score_popularity",
)

pop_summary

## 5. Logistic regression baseline

In [ ]:
X_train = train[FEATURE_COLUMNS]
y_train = train[LABEL_COLUMN].astype(int)

X_test = test[FEATURE_COLUMNS]
y_test = test[LABEL_COLUMN].astype(int)

logreg = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=500,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

start = perf_counter()
logreg.fit(X_train, y_train)
elapsed = perf_counter() - start

print(f"Training time: {elapsed:.2f} seconds")

In [ ]:
test_predictions["score_logreg"] = logreg.predict_proba(X_test)[:, 1]

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_test, test_predictions['score_logreg']):.4f}",
)
print(
    "Average Precision:",
    f"{average_precision_score(y_test, test_predictions['score_logreg']):.4f}",
)

logreg_summary, _ = ranking_metrics(
    test_predictions,
    "score_logreg",
)

logreg_summary

## 6. Histogram Gradient Boosting baseline

In [ ]:
hgb_preprocessor = SimpleImputer(strategy="median")

X_train_hgb = hgb_preprocessor.fit_transform(X_train)
X_test_hgb = hgb_preprocessor.transform(X_test)

hgb = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=200,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42,
)

start = perf_counter()
hgb.fit(X_train_hgb, y_train)
elapsed = perf_counter() - start

print(f"Training time: {elapsed:.2f} seconds")

In [ ]:
test_predictions["score_hgb"] = hgb.predict_proba(X_test_hgb)[:, 1]

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_test, test_predictions['score_hgb']):.4f}",
)
print(
    "Average Precision:",
    f"{average_precision_score(y_test, test_predictions['score_hgb']):.4f}",
)

hgb_summary, _ = ranking_metrics(
    test_predictions,
    "score_hgb",
)

hgb_summary

## 7. Compare baselines

In [ ]:
baseline_comparison = pd.DataFrame(
    [pop_summary, logreg_summary, hgb_summary]
)

baseline_comparison

## 8. Inspect one user's ranking

In [ ]:
example_user = test_predictions["user_id"].iloc[0]

example = (
    test_predictions[
        test_predictions["user_id"] == example_user
    ]
    .sort_values("score_hgb", ascending=False)
)

example[
    [
        "user_id",
        "streamer",
        "label",
        "seen_before",
        "score_popularity",
        "score_logreg",
        "score_hgb",
    ]
].head(20)

## 9. Save baseline models and predictions

In [ ]:
joblib.dump(
    logreg,
    ARTIFACT_DIR / "baseline_logistic_regression.joblib",
)

joblib.dump(
    {
        "imputer": hgb_preprocessor,
        "model": hgb,
        "feature_columns": FEATURE_COLUMNS,
    },
    ARTIFACT_DIR / "baseline_hist_gradient_boosting.joblib",
)

test_predictions.to_csv(
    PRED_DIR / "baseline_predictions.csv.gz",
    index=False,
    compression="gzip",
)

baseline_comparison.to_csv(
    ARTIFACT_DIR / "baseline_metrics.csv",
    index=False,
)

print("Saved baseline models, predictions, and metrics.")

## Next

Notebook 04 trains a PyTorch neural ranker on the same leakage-safe feature table and saves its inference scores for model comparison.